# Post-Training: SFT, Preference Optimization & RLVR

Pretraining produces a model that continues text. Everything that makes it *useful* —
instruction-following, refusals, formatting, reasoning — comes from post-training. This is
also where the field moved fastest: the RLHF-with-a-reward-model recipe that defined 2023 is
now one option among several, and no longer the default for reasoning work.

> ⏱️ Method names in this chapter are moving targets. The **shape** of the pipeline
> (imitate → prefer → verify) has been stable and is what interviews actually test.

## Why This Matters

- The three post-training stages and what each one can and cannot fix
- DPO vs RLHF vs GRPO — what changed and why the field moved
- RLVR: what makes a reward *verifiable* and why that changed reasoning training
- LoRA and QLoRA: the parameter math, and why low-rank adaptation works at all
- Distillation as a first-class deployment tool, not an afterthought
- Catastrophic forgetting, reward hacking, and the failure modes of each method

## 1. The Pipeline

```
PRETRAINING — next-token prediction on web-scale text
  → knows language, facts, code, some reasoning
  → does not follow instructions; no notion of helpful or harmful

STAGE 1: SFT — supervised fine-tuning on (instruction, response) pairs
  → follows instructions, adopts a format and persona
  → teaches IMITATION: "responses look like this"
  → cannot exceed demonstration quality

STAGE 2: PREFERENCE OPTIMIZATION — DPO / RLHF on (chosen, rejected) pairs
  → learns what humans prefer between two plausible answers
  → teaches RANKING: "this is better than that"
  → good for taste, tone, harmlessness; weak signal for correctness

STAGE 3: RLVR — RL against a verifiable checker
  → learns from outcomes an automated verifier can confirm
  → teaches CORRECTNESS: "this answer passes the test"
  → the stage that produced reasoning models
```

**Each stage exists because the previous one has a ceiling.** SFT can't beat its
demonstrations. Preference data is expensive, subjective, and noisy on questions with an
objectively right answer — two annotators comparing proofs are largely guessing. RLVR
sidesteps that by replacing the human with a checker.

## 2. Preference Optimization: RLHF → DPO → GRPO

### RLHF (the 2022–2023 recipe)
Train a reward model on human preference pairs, then optimize the policy against it with PPO.
Three models in memory (policy, reference, reward) plus a value critic. Powerful, expensive,
fiddly.

### DPO (2023)
The insight: the RL objective has a closed form that lets you skip the reward model entirely
and optimize a classification loss directly on preference pairs. No reward model, no rollouts,
no RL loop. Vastly simpler, and it matched RLHF on many alignment benchmarks — which is why it
became the default for a while.

**DPO's limitation:** it's *offline*. It learns from a fixed set of pairs someone already
collected. It never sees what the current policy would actually generate, so it can't
discover that a new behaviour is better.

### GRPO (2024 onward)
Group Relative Policy Optimization is online and critic-free. For each prompt, sample a
*group* of completions from the current policy, score them, and compute each one's advantage
relative to the **group mean**. That group-relative baseline replaces the learned value critic
— which is the expensive, unstable part of PPO.

| | RLHF/PPO | DPO | GRPO |
|---|---|---|---|
| **Reward source** | Learned reward model | Preference pairs (implicit) | Verifier or reward model |
| **Online?** | Yes | No — fixed dataset | Yes — samples during training |
| **Value critic** | Yes | N/A | No — group mean is the baseline |
| **Models in memory** | Policy, ref, reward, critic | Policy, ref | Policy, ref |
| **Best at** | General alignment | Cheap, stable alignment | Reasoning, verifiable tasks |

> 💡 **Interview Tip:** If you say "we'd use RLHF" for a math or code task, expect a follow-up.
> The current answer is RLVR with GRPO or one of its descendants. DPO remains a perfectly good
> answer for tone and harmlessness, where there is no verifier.

In [ ]:
import numpy as np
np.random.seed(0)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# ---------- DPO loss ----------
def dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1):
    """
    Offline. Operates on log-probs of a FIXED pair.
    Pushes the policy toward `chosen` relative to the reference model.
    """
    margin = (pi_chosen - ref_chosen) - (pi_rejected - ref_rejected)
    return -np.log(sigmoid(beta * margin) + 1e-12), margin

# ---------- GRPO advantages ----------
def grpo_advantages(rewards, eps=1e-8):
    """
    Online, critic-free. Given G sampled completions for ONE prompt,
    the advantage is each reward standardized within the group.
    The group mean IS the baseline — no value network required.
    """
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)

print("=== DPO on one preference pair ===")
loss, margin = dpo_loss(pi_chosen=-2.1, pi_rejected=-3.4, ref_chosen=-2.3, ref_rejected=-2.9)
print(f"  implicit reward margin : {margin:+.3f}")
print(f"  loss                   : {loss:.4f}")
print("  Positive margin = policy already prefers `chosen` more than the reference does.\n")

print("=== GRPO on a group of 8 sampled answers to one math problem ===")
# Verifier returns 1.0 if the final answer is correct, else 0.0
rewards = np.array([1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0])
adv = grpo_advantages(rewards)
print(f"  {'completion':<12}{'reward':>8}{'advantage':>12}")
for i, (r, a) in enumerate(zip(rewards, adv)):
    print(f"  {'#' + str(i):<12}{r:>8.1f}{a:>12.3f}")
print(f"\n  group solve rate: {rewards.mean():.0%}")
print("  Correct answers get positive advantage, wrong ones negative — all measured")
print("  against how THIS policy currently performs on THIS prompt. No critic needed.")

In [ ]:
# Why the group-relative baseline matters: it auto-calibrates to problem difficulty.
def show(name, rewards):
    adv = grpo_advantages(rewards)
    print(f"{name:<26} solve={np.mean(rewards):>4.0%}  adv range=[{adv.min():+.2f}, {adv.max():+.2f}]")

print("Same verifier, three difficulty regimes:\n")
show("easy (7/8 correct)",   np.array([1,1,1,1,1,1,1,0], dtype=float))
show("medium (4/8 correct)", np.array([1,1,1,1,0,0,0,0], dtype=float))
show("hard (1/8 correct)",   np.array([1,0,0,0,0,0,0,0], dtype=float))
print()
show("degenerate: all correct", np.array([1,1,1,1,1,1,1,1], dtype=float))
show("degenerate: all wrong",   np.array([0,0,0,0,0,0,0,0], dtype=float))
print()
print("The last two are the practical failure mode: when every sample in the group")
print("agrees, the advantage collapses to zero and the prompt contributes NO gradient.")
print("Curriculum design — keeping prompts near the model's current ability — is")
print("therefore not a nicety in RLVR. It is what keeps training signal alive.")

## 3. RLVR: Reinforcement Learning from Verifiable Rewards

The idea is almost embarrassingly simple: **replace the human rater with a program that
checks the answer.**

| Domain | Verifier | Reward |
|---|---|---|
| Math | Symbolic/numeric equality against ground truth | 1 if equal |
| Code | Run the unit tests | Fraction passing |
| Structured output | Schema validation | 1 if parses and validates |
| Tool use | Did the call succeed with valid arguments | 1 if valid |
| Instruction following | Programmatic constraint checks (length, format) | 1 if satisfied |

This is what unlocked reasoning models. Long chains of thought are hard to supervise
step-by-step — but if you only reward the *final* verified answer, the model is free to
discover whatever intermediate reasoning gets there. Nobody has to label the reasoning.

### The catch: reward hacking

A verifier is an optimization target, and any gap between "passes the check" and "is actually
correct" will be found and exploited. Documented failure modes include tests being trivially
short-circuited, answer-format exploits, and — most concerning — models exploiting a flaw
while producing chain-of-thought that doesn't mention it, which defeats monitoring that reads
the reasoning trace.

**Mitigations:** hold out verifiers the model never trains against, audit high-reward samples
by hand, keep a KL penalty to the reference model, and treat *any* sudden reward jump as a
suspected exploit until proven otherwise.

In [ ]:
# Reward hacking, concretely: a verifier with a gap, and what optimization does to it.
def leaky_verifier(answer: str) -> float:
    """Rewards any response that CONTAINS the right answer anywhere."""
    return 1.0 if "42" in answer else 0.0

def strict_verifier(answer: str) -> float:
    """Rewards only a correctly formatted final answer."""
    import re
    m = re.search(r"^ANSWER:\s*(-?\d+)\s*$", answer.strip(), re.M)
    return 1.0 if m and m.group(1) == "42" else 0.0

candidates = [
    ("genuine solution",      "6 times 7 is 42.\nANSWER: 42"),
    ("spam every number",     "Maybe 40 41 42 43 44 45 46 47 48"),
    ("restate the question",  "Is the answer 42? I am not sure.\nANSWER: 7"),
    ("wrong but formatted",   "The product is 36.\nANSWER: 36"),
]

print(f"{'candidate':<24}{'leaky':>8}{'strict':>8}")
print("-" * 40)
for name, text in candidates:
    print(f"{name:<24}{leaky_verifier(text):>8.0f}{strict_verifier(text):>8.0f}")

print()
print("Under the leaky verifier, 'spam every number' scores as well as a real solution")
print("at a fraction of the effort — so RL finds it. The model is not cheating; it is")
print("doing exactly what you rewarded. Verifier design IS reward design.")

## 4. Parameter-Efficient Fine-Tuning

### LoRA
Freeze the pretrained weight $W$ and learn a low-rank update: $W' = W + \frac{\alpha}{r}BA$,
where $B \in \mathbb{R}^{d_{out} \times r}$ and $A \in \mathbb{R}^{r \times d_{in}}$ with
$r \ll d$.

The hypothesis: the *update* needed for adaptation has low intrinsic rank, even though $W$
itself does not. Empirically, $r = 8$–$64$ matches full fine-tuning on most adaptation tasks.

Three practical properties worth naming in an interview:
1. **Base weights never move**, so catastrophic forgetting is structurally prevented.
2. **Adapters are swappable** — serve one base model with many task adapters, hot-swapped per
   request. This is a serving-cost argument, not just a training one.
3. **Adapters can be merged** into $W$ post-training for zero inference overhead — at the cost
   of losing swappability.

### QLoRA
Quantize the frozen base to 4-bit and train LoRA adapters on top in higher precision.
Gradients flow *through* the quantized base without updating it. This is what makes
fine-tuning a large model on a single GPU feasible. You pay a little quality and some training
speed for a large drop in memory.

In [ ]:
# The parameter math — this is the number interviews ask for.
def lora_params(d_in, d_out, rank):
    return rank * (d_in + d_out)

def report(name, d_in, d_out, n_matrices, rank):
    full = d_in * d_out * n_matrices
    lora = lora_params(d_in, d_out, rank) * n_matrices
    print(f"{name:<30} {full/1e6:>10.1f}M {lora/1e6:>10.2f}M {100*lora/full:>9.2f}%")

print(f"{'target':<30} {'full FT':>11} {'LoRA r=16':>11} {'trainable':>10}")
print("-" * 64)
report("Attention q,v (32 layers)", 4096, 4096, 32 * 2, 16)
report("Attention q,k,v,o (32 lyr)", 4096, 4096, 32 * 4, 16)
report("+ FFN (32 layers)",         4096, 11008, 32 * 3, 16)

print()
print("Memory during training (7B model, rough):")
for label, gb in [
    ("Full FT, Adam, bf16", 7 * 2 + 7 * 2 + 7 * 8),   # weights + grads + optimizer states
    ("LoRA, bf16 base",      7 * 2 + 0.1),
    ("QLoRA, 4-bit base",    7 * 0.5 + 0.1),
]:
    print(f"  {label:<24} ~{gb:>5.1f} GB")

print()
print("Full fine-tuning is dominated by Adam's two moment buffers (4 bytes/param each),")
print("not by the weights. Freezing the base removes gradients AND optimizer state,")
print("which is why LoRA's saving is far larger than the trainable-parameter ratio suggests.")

## 5. Distillation

Train a small **student** to imitate a large **teacher**. Chronically under-taught relative to
how much it's used in production.

| Flavour | Signal | Notes |
|---|---|---|
| **Response distillation** | Teacher's generated outputs as SFT targets | Simplest; just a data-generation pipeline |
| **Logit distillation** | Teacher's full output distribution | Richer signal; needs teacher logits, so open weights or self-hosted |
| **Reasoning distillation** | Teacher's chain of thought plus the answer | How small reasoning models are built |

**Why it belongs in an MLE's toolkit:** most production LLM features do not need a frontier
model. They need frontier *quality on one narrow task*. Distillation is the standard way to
convert the first into the second — often 10–50× cheaper per request at comparable task
quality.

**The standard pipeline:** prototype with the frontier model → log real production traffic →
filter and verify the good responses → fine-tune a small model on them → A/B the small model
against the large one → route the easy majority to the small model and escalate the hard tail.

**Caveats:** the student inherits the teacher's mistakes and biases; quality is capped near
the teacher on the distilled distribution and degrades off it; and check the teacher's terms
of service before distilling from a commercial API.

## 6. Catastrophic Forgetting

Fine-tuning on a narrow task degrades unrelated capabilities. It is the single most common
way a fine-tuning project quietly fails: the target metric improves, ships, and something
else regresses that nobody was measuring.

| Mitigation | Mechanism | Cost |
|---|---|---|
| **LoRA / QLoRA** | Base weights physically cannot move | Slight capacity limit |
| **Replay** | Mix general data into the fine-tuning set | Needs representative general data |
| **Low LR + early stopping** | Move less far from the initialization | Blunt; may underfit the target |
| **KL penalty to reference** | Explicitly penalize divergence | An extra forward pass per step |
| **Broad eval suite** | Detect it rather than prevent it | The one you must do regardless |

> 💡 **Interview Tip:** The strongest answer here isn't a technique, it's a process: *"I'd keep
> a general-capability eval suite and run it on every fine-tuned checkpoint, not just the task
> metric."* Forgetting is invisible unless you look for it.

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

# Catastrophic forgetting, minimal reproduction.
# Task A = "general capability", Task B = "the narrow thing you fine-tune on".
XA, yA = make_classification(n_samples=600, n_features=12, n_informative=8, random_state=1)
XB, yB = make_classification(n_samples=600, n_features=12, n_informative=8, random_state=7)

def evaluate(m):
    return accuracy_score(yA, m.predict(XA)), accuracy_score(yB, m.predict(XB))

# --- Naive sequential fine-tuning ---
m = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.05, random_state=0)
m.partial_fit(XA, yA, classes=[0, 1])
for _ in range(30):
    m.partial_fit(XA, yA)
a0, b0 = evaluate(m)

for _ in range(30):
    m.partial_fit(XB, yB)          # fine-tune on B only
a1, b1 = evaluate(m)

# --- Same budget, but with replay (mix 50% of A back into each step) ---
mr = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.05, random_state=0)
mr.partial_fit(XA, yA, classes=[0, 1])
for _ in range(30):
    mr.partial_fit(XA, yA)

rng = np.random.default_rng(0)
for _ in range(30):
    idx = rng.choice(len(XA), size=len(XA) // 2, replace=False)
    Xmix = np.vstack([XB, XA[idx]])
    ymix = np.concatenate([yB, yA[idx]])
    mr.partial_fit(Xmix, ymix)
a2, b2 = evaluate(mr)

print(f"{'stage':<34}{'Task A':>9}{'Task B':>9}")
print("-" * 52)
print(f"{'after pretraining on A':<34}{a0:>9.3f}{b0:>9.3f}")
print(f"{'after naive fine-tune on B':<34}{a1:>9.3f}{b1:>9.3f}")
print(f"{'after fine-tune on B w/ replay':<34}{a2:>9.3f}{b2:>9.3f}")
print()
lost = a0 - a1
print(f"Naive fine-tuning gave up {lost:.3f} accuracy on Task A to gain {b1 - b0:.3f} on Task B.")
print(f"Replay recovered {a2 - a1:.3f} of that loss ({(a2 - a1) / lost:.0%}) at the same step budget,")
print(f"while still reaching {b2:.3f} on Task B.")
print()
print("A linear model is the mildest possible version of this. In a large network with")
print("billions of shared parameters the effect is far more severe — and you only see it")
print("if you are still measuring Task A after you stop caring about it.")

## Common Interview Questions

**Q: Walk me through modern post-training.**
Three stages. SFT teaches imitation from demonstration pairs — it establishes format and
instruction-following but can't exceed its demonstrations. Preference optimization (DPO, or
RLHF where you want an inspectable reward model) teaches ranking from chosen/rejected pairs,
which is right for tone and harmlessness. RLVR teaches correctness by optimizing against an
automated verifier — the stage that produced reasoning models. Each exists because the
previous one has a ceiling.

**Q: DPO vs GRPO — when would you pick each?**
DPO is offline and learns from a fixed preference dataset: cheap, stable, and a good fit when
"better" is subjective and no verifier exists. GRPO is online — it samples a group of
completions from the current policy and uses the group mean as the advantage baseline instead
of a learned critic. Use GRPO when you have a programmatic verifier and want the model to
*discover* better behaviours rather than imitate ranked ones. Math, code, and tool use are the
natural fits.

**Q: Why is GRPO critic-free, and why does that matter?**
PPO needs a value network to estimate the baseline for advantage computation — that critic is
roughly as large as the policy, costs memory, and is a common source of instability. GRPO
gets the baseline for free by sampling G completions per prompt and standardizing rewards
within the group. Fewer models in memory, one less thing to diverge.

**Q: What's the biggest risk in an RLVR setup?**
Reward hacking. The verifier is the objective, so any gap between "passes the check" and "is
correct" gets found and exploited — and the model may not verbalize the exploit in its chain
of thought, which defeats trace-based monitoring. Mitigations: held-out verifiers, manual
audit of high-reward samples, a KL penalty to the reference policy, and treating sudden reward
jumps as suspected exploits.

**Q: LoRA saves ~99% of trainable parameters. Does it save 99% of memory?**
No, but it saves more than you'd guess — for a different reason. Full fine-tuning memory is
dominated by Adam's optimizer states (two fp32 moments per parameter), not the weights.
Freezing the base eliminates gradients *and* optimizer state for those parameters, so the drop
is large. But the frozen base still has to be resident, which is exactly what QLoRA addresses
by quantizing it to 4-bit.

**Q: When would you distil instead of fine-tune?**
When quality is already acceptable and the problem is cost or latency. Fine-tuning adapts a
model to a task; distillation transfers existing capability into a smaller, cheaper model. The
production pattern is to prototype with a frontier model, log and filter real traffic, train a
small student on it, then route the easy majority to the student and escalate the hard tail.

**Q: How do you know fine-tuning didn't break something else?**
You don't, unless you measure. Keep a general-capability eval suite separate from the task
metric and run it on every checkpoint. Catastrophic forgetting doesn't announce itself — the
target metric goes up while unrelated behaviour silently regresses.

## Key Takeaways
- Pipeline shape is stable: SFT (imitate) → preference optimization (rank) → RLVR (verify)
- DPO is offline and critic-free; GRPO is online and critic-free, using the group mean as baseline
- RLVR replaced human raters with programmatic verifiers — the mechanism behind reasoning models
- Verifier design *is* reward design: any gap between "passes" and "correct" will be exploited
- Degenerate groups (all-right or all-wrong) give zero gradient, so curriculum matters in RLVR
- LoRA's memory win comes mostly from eliminating Adam optimizer state, not from the weights
- QLoRA quantizes the frozen base to 4-bit so large models fine-tune on a single GPU
- Distillation is the standard cost lever: prototype large, log traffic, train small, route by difficulty
- Always run a general eval suite alongside the task metric — forgetting is invisible otherwise